# Part 2 · A mini crystal structure prediction  (~9 min)

Generate trial packings, relax each one, rank by energy. We start from the paracetamol molecule only and look for
Form I and Form II. Packings come from [PyXtal](https://github.com/MaterSim/PyXtal), energies from MACE-OFF.

In [ ]:
# @title 0 · Setup  (run this cell first; ~1-2 min on Colab)
import os, sys
REPO = "https://github.com/fmggggg/agentic-ai-for-functional-materials-discovery"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/csp-tutorial"):
        !git clone -q {REPO} /content/csp-tutorial
    %cd /content/csp-tutorial
    !pip -q install pyxtal mace-torch py3Dmol pymatgen rdkit "requests==2.32.4"
import csp_utils as cu
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("ready")

## 2.1 Take the molecule (keep its conformer rigid)

In [ ]:
form1 = cu.load_cif(cu.DATA / "paracetamol_form_I.cif")
mol = cu.molecule_from_crystal(form1)
print(mol.composition, len(mol), "atoms")

## 2.2 Generate random crystals

Most organic crystals are in one of five space groups: P2₁/c, P1̄, P2₁2₁2₁, C2/c, P2₁.

In [ ]:
trial = cu.random_crystal(mol, sg=14, z=4, seed=1)   # P2_1/c, 4 molecules
print("density of this random guess:", round(cu.density(trial), 3), "g/cm3  (experiment: 1.33)")
cu.show(trial, supercell=(2, 2, 2))

In [ ]:
import time
plan = [(14, 4), (2, 2), (19, 4), (4, 2), (61, 8)]
t0 = time.time()
raw = [cu.random_crystal(mol, sg, z, seed=i) for i, (sg, z) in enumerate(plan * 6)]
print(f"{len(raw)} structures in {time.time()-t0:.1f} s")
plt.hist([cu.density(a) for a in raw], bins=15, color="#C9A27E")
plt.axvline(cu.density(form1), color="#893F0D", lw=2, label="Form I (exp)")
plt.xlabel("density (g/cm$^3$)"); plt.ylabel("count"); plt.legend(); plt.title("Random guesses are loosely packed");

## 2.3 Relax two of them

Atoms and cell both move. We stop after 100 steps, so these are not fully converged.

In [ ]:
res = []
for k in range(2):
    t0 = time.time()
    at = cu.relax(raw[k], steps=100)
    res.append(dict(id=k, sg=plan[k % 5][0], density_before=round(cu.density(raw[k]), 3),
                    density_after=round(cu.density(at), 3),
                    E_per_mol=round(cu.energy_per_molecule(at), 1), seconds=round(time.time() - t0)))
    print(res[-1])
pd.DataFrame(res)

> **Question 3.** A real CSP study relaxes 10⁴–10⁵ structures per molecule. At your speed, how long would that take?

## 2.4 The full landscape (precomputed)

About 220 random structures, relaxed beforehand. Matches to the experimental forms are marked.

In [ ]:
L = cu.load_landscape("landscape_random")
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(L.density, L.rel_energy_kjmol, s=8, c="#BDBDBD", label="random search")
for col, mk, lab in [("rms_form_I", "*", "matches Form I"), ("rms_form_II", "D", "matches Form II")]:
    hit = L[L[col].notna()]
    ax.scatter(hit.density, hit.rel_energy_kjmol, s=90, marker=mk, edgecolor="k", label=f"{lab} ({len(hit)})")
ax.set_xlabel("density (g/cm$^3$)"); ax.set_ylabel("relative energy (kJ/mol)"); ax.set_ylim(-1, 40)
ax.legend(); ax.set_title("Paracetamol CSP landscape (MACE-OFF)");

In [ ]:
L = L.sort_values("rel_energy_kjmol").reset_index(drop=True)
for col in ["rms_form_I", "rms_form_II"]:
    idx = L.index[L[col].notna()]
    print(col, "-> best rank:", (idx.min() + 1) if len(idx) else "not found",
          "| E_rel =", round(L.loc[idx.min(), "rel_energy_kjmol"], 2) if len(idx) else "-")
print("structures within 5 kJ/mol of the global minimum:", (L.rel_energy_kjmol < 5).sum())

> **Question 4.** Are the experimental forms at the bottom? How many other structures are within 5 kJ/mol?
>
> **Question 5.** What fraction of the random structures ended up as Form I?

In [ ]:
for col in ["rms_form_I", "rms_form_II"]:
    print(col, f"hit rate = {L[col].notna().mean()*100:.2f} %")